# CRUD, formulaires, gestions d'utilisateurs


In [10]:
# on restaure notre base de données en remplaçant la BDD par son backup 
!cp ./richelieu.db.bak ./richelieu.db

On sait maintenant:
- créer une appli Flask
- la connecter à une base de données via SQLAlchemy  
- modéliser une BDD via l'ORM SQLAlchemy
- parcourir les objets de l'ORM, et leurs relations
- faire des requêtes `SELECT` avec SQLAlchemy
- créer des templates Jinja

**En bref**: tout pour faire un site "catalogue", qui affiche des données sans pouvoir .

Aujourd'hui, on va voir comment faire **toutes les opérations CRUD** sur notre BDD directement depuis le site. On va donc apprendre à:
- **faire des requêtes `CREATE`, `UPDATE` et `DELETE`** avec SQLAlchemy (on sait déjà faire le `READ`, on saura donc faire toutes les opérations CRUD) 
- **créer des formulaires HTML** pour ajouter/modifier/supprimer des données depuis le site

---

# Apparté bonnes pratiques

On code maintenant ensemble depuis 8h et on va commencer à voir des fonctions un peu plus longues.

Rappelez vous que **vous codez pour le long terme**: votre code devra être utilisé par vous et par d'autres, pendant plusieurs années. Retenez que c'est **plus dur de lire du code que d'en écrire** (il faut s'adapter à une autre logique, traduire le code en langage humain...), et c'est **beaucoup plus dur de lire du code de quelqu'un d'autre** (il faut s'adapter à la logique de quelqu'un d'autre). **Pour coder, il faut avoir beaucoup d'informations en tête** (noms de variables, ce qu'elles contiennent, ce qu'on veut en faire, comment...). 

Votre but, c'est d'être **le plus descriptif pour réduire le volume d'informations que vous devez retenir** dans votre "RAM mentale".

**Voici donc quelques bonnes pratiques à adopter**:
- **commentez votre code**: chaque fonction doit avec une docstring
- **une fonction = une opération**: une fonction doit faire une chose, et le faire bien. Plus une fonction est longue, 
    - plus elle est complexe 
    - dure à comprendre
    - plus il y a un risque d'erreur
- **faites des fonctions courtes**: une fonction doit faire au grand maximum la hauteur de votre écran (format paysage)  
- **soyez explicites** dans nos noms de variables, de fonctions et de classes:
    - une variable appelée `x` ou une fonction `f`, ça ne dit pas grand chose sur ce que contient la variable, ou ce que la fonciton fait
    - **votre objectif**:
        - vos variables décrivent leur contenu
        - vos noms de fonction décrivent l'opération faite par la fonction 
        - écrire du code synthétique, c'est moins important qu'écrire du code compréhensible !
- **typez vos fonctions**: peut-être le plus important. **Typer, c'est expliciter ce que contient une variable, et ce que fait une fonction**. C'est **très douloureux** de devoir mettre des prints partout pour comprendre ce que contiennent des variables, et les type hints permettent d'alléger beaucoup ça.

En bref, 
```py
# cette fonction
def f(a, b):
    if len(a) > b:
        return f"{a[:b]}[...]"
    return a

# est beaucoup moins claire que:
def raccourcir(string, max_len):
    if len(string) > max_len:
        return f"{string[:max_len]}[...]"
    return string

# qui est moins claire que:
def raccourcir_avec_types(string: str, max_len: int) -> str:
    if len(string) > max_len:
        return f"{string[:max_len]}[...]"
    return string
```



---

# Sécurisation de l'application: la `SECRET_KEY`

En interne, Flask doit confirmer l'authenticité de plein de choses. Dans ce qui va suivre, Flask a par exemple besoin de confirmer que les données d'un formulaire reçues sont authentiques. Pour cela, Flask a besoin que l'on définisse une `SECRET_KEY`.

**On définit `SECRET_KEY`** dans `app/utils/constants.py`:
```py
from warnings import warn

# la clé top secrète
secret_key_default = "Une clé secrète"
SECRET_KEY = "Une clé secrète"

if SECRET_KEY == secret_key_default:
    warn(f"Changez votre clé secrète avant de passer en production ! Clé secrète actuelle: {SECRET_KEY}")
```

Et dans `app/app.py`, **on ajoute `SECRET_KEY` à la configuration de notre appli** Flask:

```py
app = Flask(
    APP_NAME,
    template_folder=DIR_TEMPLATES, 
    static_folder=DIR_STATICS
)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{PATH_DB}"
app.config["SECRET_KEY"] = SECRET_KEY
db = SQLAlchemy(app)
```

**À noter**: une `SECRET_KEY` doit toujours être confidentielle. C'est pourquoi dans `constants.py`, on voit un `warn()`. **Avant de mettre une appli en production, il faut toujours définir une clé secrète *random*** via un algorithme solide.

> **Lancez l'appli `apps/s5/secret_key` et regardez le warning qui s'affiche (le reste de l'appli n'a pas changé)**
> ```py
> python apps/s5/secret_key/main.py
> ```


---

# Première requête `CREATE`

On va maintenant apprendre à **créer des nouvelles ressources iconographiques**.

TODOOOOOOOOOOOOOOO: changer la suite du cours pour faire du CRUD sur la table Iconography.

In [11]:
# NOTE: pour le notebook, je combine dans un seul bloc la définition
# de l'appli et tous les modèles de la base de données
# on a donc déjà vu tout le code en dessous.

from typing import List, Optional, Dict
from pathlib import Path

from flask import Flask
from flask_sqlalchemy import SQLAlchemy
from werkzeug.security import generate_password_hash
from sqlalchemy import ForeignKey, JSON
from sqlalchemy.orm import Mapped, mapped_column, relationship

path_to_db = Path("./richelieu.db").absolute()
print(path_to_db)

APP_NAME = "Catalogue Richelieu"
app = Flask(APP_NAME)
app.config["SQLALCHEMY_DATABASE_URI"] = f"sqlite:///{path_to_db}"
db = SQLAlchemy(app)


class IconographyPlace(db.Model):
    __tablename__ = "iconography_place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_place: Mapped[int] = mapped_column(ForeignKey("place.id"))


class IconographyTheme(db.Model):
    __tablename__ = "iconography_theme"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    id_iconography: Mapped[int] = mapped_column(ForeignKey("iconography.id"))
    id_theme: Mapped[int] = mapped_column(ForeignKey("theme.id"))


class Author(db.Model):
    __tablename__ = "author"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    author_name: Mapped[str] = mapped_column(unique=True)

    iconography: Mapped[List["Iconography"]] = relationship(
        back_populates="author", 
    )


class Theme(db.Model):
    __tablename__ = "theme"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    theme_name: Mapped[str] = mapped_column(unique=True)
    richelieu_url: Mapped[str] = mapped_column(unique=True)

    iconography: Mapped[List["Iconography"]] = relationship(
        secondary=IconographyTheme.__table__,
        back_populates="theme"
    )


class Place(db.Model):
    __tablename__ = "place"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    address: Mapped[Optional[str]]
    richelieu_url: Mapped[str] = mapped_column(unique=True)
    loc: Mapped[Dict] = mapped_column(JSON)
    plot: Mapped[Dict] = mapped_column(JSON)
    date_lower: Mapped[int]
    date_upper: Mapped[int]

    iconography: Mapped[List["Iconography"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="place"
    )


class Iconography(db.Model):
    __tablename__ = "iconography"
    
    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    title: Mapped[str]
    iiif_manifest_url: Mapped[str]
    iiif_image_url: Mapped[str]
    source_url: Mapped[Optional[str]]
    richelieu_url: Mapped[str]
    date_lower: Mapped[Optional[int]]
    date_upper: Mapped[Optional[int]]
    institution: Mapped[str]
    id_author: Mapped[Optional[int]] = mapped_column(ForeignKey("author.id"))
    
    author: Mapped[Optional["Author"]] = relationship(back_populates="iconography")
    theme: Mapped[List["Theme"]] = relationship(
        secondary=IconographyTheme.__table__,
        back_populates="iconography"
    )
    place: Mapped[List["Place"]] = relationship(
        secondary=IconographyPlace.__table__,
        back_populates="iconography"
    )


/home/paul/Documents/cours/tnah_devapp/richelieu.db


### Créer un nouvel objet `Iconography` et le sauvegarder

#### 1. Créer un objet `Iconography` côté Python

Pour créer un `Iconography`, rien de plus facile. Nos modèles de base de données sont des classes, donc `Iconography` est une classe et **chaque ressource iconographique est une instance de cette classe**.

On peut trouver des plusieurs ressources iconographiques à ajouter dans [`./data/iconography_sample.md`](./data/iconography_sample.md) 

In [12]:
new_icono = Iconography(
    title="Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]",
    iiif_manifest_url="https://gallica.bnf.fr/iiif/ark:/12148/btv1b530124015/manifest.json",
    iiif_image_url="https://gallica.bnf.fr/iiif/ark:/12148/btv1b530124015/f1/full/1000/0/native.jpg",
    source_url="https://gallica.bnf.fr/ark:/12148/btv1b530124015",
    richelieu_url="https://quartier-richelieu.inha.fr/iconographie/qr1703e89942a0c4fd08052e91956f67719",
    institution="Bibliothèque nationale de France",
    date_lower=1807,
    date_upper=1808,
    id_author=235,
) 
print(new_icono)
print("title:", new_icono.title)
print("date_lower:", new_icono.date_lower)
print("id_author:", new_icono.id_author)


<Iconography (transient 139451821488224)>
title: Partant pour la Syrie [ ] Paroles et Musique de La Reine Hortense Paris, Colombier, Editeur, Rue Vivienne, 6 : [estampe]
date_lower: 1807
id_author: 235


**Explication de code**: 
- `Iconography()` permet de créer une instance de classe
- les arguments qu'on passe à `Iconography()` sont les valeurs qu'on veut donner à cette instance (équivalent SQL: les valeurs que l'on donne à chaque colonne)

#### 2. Le sauvegarder en base

Pour le moment, `new_iconography` n'a pas été sauvegardé. En fait, aucune interaction avec la base de données n'a encore eu lieu.

**Il faut faire un `commit` dans la base de données** pour enregistrer `new_icono`:

In [ ]:
# pour rappel, on doit utiliser app.app_context dans les notebooks parce qu'on est hors d'une application Flask. 
with app.app_context():
    db.session.add(new_icono)
    db.session.commit()
    # on remarque maintenant que `new_icono` a un ID. 
    print(new_icono)
    new_icono_id = new_icono.id


<Iconography 1311>
1311


**Explication de code**:
- `db.session.add()`: on ajoute `new_icono` à la session pour qu'il soit sauvegardé
- `db.session.commit()` sauvegarde tous les changements réalisés pendant une session (create/update/delete).

Le `commit`, si vous vous rappelez, est à la base du SQL: toutes les interactions avec une base de données ont lieu dans une **transaction**, qui est composée de une ou plusieurs requêtes SQL et qui se termine par un **commit**, qui sauvegarde tous les changements de base de données qui ont eu lieu pendant la transaction.

## Créer un utilisateur depuis `Iconography`:  `create_iconography`

On a vu comment créer un objet `Iconography` et le sauvegarder en base. On va maintenant:
- **créer une fonction `create_iconography`** qui gère la création d'objets iconographiques. `create_iconography` prend en paramètres toutes les valeurs que l'on veut donner à notre ressource iconographiques, crée le nouvel objet `Iconography` et le sauvegarde en base.
- **ajouter `create_iconography` à la classe `Iconography`**

Regardons bien l'exemple ci-dessous:

```py
class User(db.Model):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]

    @staticmethod
    def create_user(user_mail: str, user_name: str, user_password: str) -> Tuple[bool, Union["User", str]]:
        """
        créer un nouveau user.

        notre fonction retourne:
        - (True, User) en cas de succès 
        - (False, <message d'erreur>) en cas d'erreur
        donc, le 1er item permet de savoir si l'insertion a fonctionné 
        """
        # liste de nos erreurs
        errors = []

        # 1. on vérifie que l'utilisateur.ice a fourni toutes les données
        if user_mail is None:
            errors.append("Veuillez fournir un email")
        if user_name is None:
            errors.append("Veuillez fournir un nom d'utilisateur")
        if user_password is None:
            errors.append("Veuillez fournir un mot de passe")

        # on vérifie si il existe un autre `user` avec le même mail
        existing_user = db.session.execute(
            db.select(User).filter(User.user_mail == user_mail)
        ).scalars().all()
        if len(existing_user):
            errors.append(f"Un utilisateur existe déjà pour le mail: {user_mail}")

        # il y a eu des erreurs => pas d'insert
        if len(errors):
            return False, errors

        # pas d'erreurs => on fait l'insert
        new_user = User(
            user_name=user_name,
            user_mail=user_mail,
            user_password=generate_password_hash(user_password)
        ) 
        try:
            db.session.add(new_user)
            db.session.commit()
        except Exception as e:
            # en cas d'erreur au moment de l'insert, on retourne False et le message d'erreur de l'appli
            print(e)
            return False, str(e)
```

**Ce qu'on a ajouté à la création d'utilisateurs vue au dessus**:
- **des vérifications**:
    - une vérification de l'*user input*
    - une vérification pour savoir si un compte existe déjà avec ce mail
- **un `try...except` autour du `commit()`**: la base de données SQL a son propre système de vérifications, et donc des problèmes inattendus peuvent émerger au moment de l'insertion, donc on gère proprement cette erreur.

**`create_user` est une `staticmethod` de la classe `User`**:
- une `staticmethod` est une méthode de `User` qui n'hérite pas d'une instance de la classe (`self`). Elle n'agit donc pas sur un objet `User`, et **sa présence dans la classe est juste une question d'organisation de code**.
- pour **utiliser `User.create_user`**, on fait: `user = User.create_user(...)`

## Créer un `User` depuis l'application

On a maintenant une classe `User`, et une méthode `User.create_user()` qui permet de créer des nouveaux comptes utilisateur.ice.s. **Pour compléter, il nous manque**:
- **une route** pour créer un utilisateur
- **une page HTML** avec un formulaire qui permet de créer son compte.

### Organisation du code

On ajoute les fichiers suivants à notre application (`./apps/s5/insert/`):

```txt
└── app
    ├── app.py
    ├── models
    │   ├── forms.py  # tout le code python pour créer des formulaires
    │   └── users.py  # nos `db.Models` relatifs à la gestion d'utilisateur.ice.s
    ├── routes
    │   └── users.py  # les routes relatives à la gestion d'utilisateur.ice.s 
    └── templates
        └── pages
            └── user_create.html  # template HTML pour créer un utilisateur.
```

### Créer des formulaires avec WTForms

Les formulaires, c'est un peu la plaie du développement Web: il faut recevoir des données des utilisateur.ice.s, et donc les valider (par exemple, vérifier que les champs obligatoires sont remplis, un mail ou une date suit le bon format). En plus de ça, il faut garantir l'accessibilité (et donc s'y connaître en accessibilité HTML). Ça peut vite devenir complexe et douloureux à gérer, et c'est répétitif.

On peut créer nos formulaires à la main (en faisant des `<form>` HTML et gérant la validation à la main). Cependant, **des librairies implémentent déjà toute la logique dont on a besoin pour créer des formulaires**.

On va donc apprendre à utiliser **[WTForms](https://wtforms.readthedocs.io/en/3.2.x/), et [Flask-WTF](https://flask-wtf.readthedocs.io/en/1.2.x/)** (plugin pour intégrer WTForms à Flask).

Avec WTForms, on a besoin de 3 choses: 
- **une classe Python** qui définit le formulaire
- **une template HTML** qui sera l'interface utilisateur pour le formulaire
- **une route** qui permette d'accéder au formulaire et de soumettre le formulaire.

#### La classe `UserCreateForm`

On l'a dit, [`app/models/forms.py`](`./apps/s5/insert/app/models/forms.py`) stockera toutes nos classes WTForms.

Pour rappel, pour créer un nouveau `user` on a besoin d'un `user_mail`, d'un `user_name` et d'un `user_password`. Donc, **notre formulaire aura 3 champs obligatoires**.

**Voici `UserCreateForm`**, qui permet de créer un formulaire.

```py
from flask_wtf import FlaskForm
from wtforms import StringField, PasswordField
from wtforms.validators import DataRequired, Email, Length

class UserCreateForm(FlaskForm):
    user_name = StringField("Nom d'utilisateur.ice", validators=[DataRequired(), Length(max=50)])
    user_mail = StringField("Email", validators=[DataRequired(), Email()])
    user_password = PasswordField("Password", validators=[DataRequired(), Length(min=5, max=50)])
```

C'est assez simple:
- **notre formulaire est représenté par une classe Python qui hérite de `FlaskForm`**: `class UserCreateForm(FlaskForm)`
- **chaque champ du formulaire est une propriété de la classe** `UserCreateForm`
- **chaque champ est associé à un `Field`**: `StringField`, `PasswordField`... Le `Field`, c'est le type de champ. WTForms en définit [un certain nombre](https://wtforms.readthedocs.io/en/3.2.x/fields/#basic-fields). Chaque `Field` prend plusieurs arguments:
    - **en 1er, le nom du champ** qui sera affiché à l'utilisateur
    - **l'argument `validators`** définit une liste avec toutes les règles de validation qui seront associées à ce champ. `user_name` est obligatoire et mesure au maximum 50 caractères, `user_mail` est obligatoire et doit être un mail... Il y a [beaucoup de validateurs possibles](https://wtforms.readthedocs.io/en/3.2.x/validators/).

Tous les formulaires qu'on créera suivront la même logique.

#### La template `user_create.html`

La template HTML [`app/templates/pages/user_create.html`]() permet de **créer une interface utilisateur pour le formulaire**, via une template HTML. Voici son contenu:

```html
{% block main_content %}
    <h1 class="title">Créer un compte utilisateur</h1>

    <!-- 
        method="POST": l'envoi du formulaire correspond à une requête HTTP POST
        action="{{ url_for('user_create') }}": on ira taper sur la route `user_create` quand on enverra les données du formulaire
     -->
    <form method="POST" action="{{ url_for('user_create') }}">
        <!-- obligatoire pour qu'un champ soit valide, permet de garantir l'intégrité des données -->
        {{ form.csrf_token }} 
        <dl>
            <!-- on intère sur toutes les propriétés de notre classe `UserCreateForm`: 
                user_name, user_mail, user_password -->
            {% for field in form %}
                <!-- CSRFTokenField est le form.csrf_token, déjà présent au dessus => on le masque ici -->
                {% if field.type != 'CSRFTokenField' %}
                    <!-- le titre du champ. field.label, c'est le 1er argument de nos `Fields` définis dans `UserCreateForm` -->
                    <dt>
                        {{ field.label }}
                        {% if field.flags.required %}*{% endif %}
                    </dt>
                    <!-- l'input utilisateur, c'est {{ field }}. le HTML valide est généré par 
                        WTForms en fonction du type de `Field`. -->
                    <dd>
                        {{ field }}
                        <!-- si il y a des erreurs pour un champ, on affiche l'erreur en dessous de ce champ 
                            (p.ex: notre format d'email n'est pas valide) -->
                        {% if field.errors %}
                            <ul class="errors">
                                {% for error in field.errors %}
                                    <li>{{ error }}</li>
                                {% endfor %}
                            </ul>
                        {% endif %}
                    </dd>
                {% endif %}
            {% endfor %}
        </dl>
        <!-- input type="submit", c'est le bouton qui permet d'envoyer un formulaire -->
        <input type="submit" value="Créer" class="button is-rounded negative">
    </form>
{% endblock %}
```

**En résumé**:
- **on crée un `<form>`** qui définit la méthode HTTP et la route associée au formulaire 
- **on intère sur tous les champs** définis dans notre classe `UserCreateForm`
- **on finit par un `<input>`** qui permet de confirmer l'envoi du formulaire.
- tout le reste est géré par WTForms.

À noter: l'utilisation de `<form>` et de `<input>` n'est pas propre à WTForms et se retrouve dans tous les formulaires.

#### La route `user_create`

On a maintenant une classe formulaire et une template pour ce formulaire.

Maintenant, on va **créer la route `user_create` pour accéder au formulaire et pouvoir le soumettre**. Cette route se trouve dans [`app/routes/users.py`](./apps/s5/insert/app/routes/users.py)

```py
@app.route("/user/nouveau/", methods=["GET", "POST"])
def user_create():
    form = UserCreateForm()

    # form.validate_on_submit est True si:
    # - la requête est POST (on a soumis un formulaire)
    # - le formulaire est valide (wtforms a bien validé toutes les données fournies)
    if form.validate_on_submit():
        # on récupère les données et on les passe à User.create_user
        # `.data` permet de sélectionner la valeur fournie par l'utilisateur.ice
        user_name = form.user_name.data
        user_mail = form.user_mail.data
        user_password = form.user_password.data
        # `create_user` retourne:
        # - un booleen qui indique si la création réussi
        # - soit l'objet User crée, soit une liste d'erreurs
        success, data = User.create_user(
            user_name=user_name, 
            user_mail=user_mail, 
            user_password=user_password
        )
        # l'insert a réussi => rediriger sur la page d'accueil
        if success:
            flash("Compte utilisateur créé avec succès ! Vous pouvez maintenant vous connecter.", "success")
            return redirect("/")
        # l'insert a échoué => afficher les messages d'erreur.
        else: 
            data = "Les erreurs suivantes ont été repérées:" + ", ".join(data)
            flash(data, "error")
            return  render_template("pages/user_create.html", form=form, app_name=APP_NAME)
    return render_template("pages/user_create.html", form=form, app_name=APP_NAME)
```

**Explication de code**:
- **notre route accepte 2 méthodes HTTP**: cela est défini dans le `@app.route()` avec `methods=["GET", "POST"]`
    - `GET` est utilisé pour **accéder au formulaire** sans soumettre de données
    - `POST` est utilisé pour **soumettre le formulaire** (rappelez vous de la template, où on voit `<form method="POST">`)
- **la route a 2 branches correspondantes**
    - `if form.validate_on_submit()` confirme que on a envoyé une requête `POST` et que le formulaire est valide **=> on tente une insertion** avec `User.create_user()`
    - sinon, (c'est une requête `GET` pour accéder au formulaire ou les données ne sont pas valides), **on renvoie le formulaire** `user_create.html`.
- **`flash` est utilisé pour afficher les messages de succès ou d'erreur**. flash est une fonction Flask qui prend en 1er argument les messages à afficher, en 2e argument le statut du message (`success` ou `error`).

Pour afficher les messages flashés, **on ajoute à `base.html` le bloc suivant**:

```html
{% with messages = get_flashed_messages(with_categories=True) %}
    {% if messages %}
        <div class="flex-center">
            <ul class=flashes>
                {% for category, message in messages %}
                    <li class=
                        {% if category=="error" %}
                            "flash-message errors"
                        {% else %}
                            "flash-message success"
                        {% endif %}
                    >{{ message }}</li>
                {% endfor %}
            </ul>
        </div>
    {% endif %}
{% endwith %}
```

Et enfin, **on importe les routes de `app/routes/users.py`** dans `app/app.py` en modifiant la dernière ligne du fichier:

```py
# l'ancien import était: `from app.routes import generic`
from app.routes import generic, users
```

### En résumé

Créer un formulaire, c'est:
- créer une classe `WTForms`
- créer une template HTML pour ce formulaire
- créer une route qui permette d'accéder au formulaire avec `GET` et de soumettre les résultats avec `POST`.

> **Lancer l'application `apps/s5/insert`**:
> ```py
> python ./apps/s5/insert/main.py
> ```

**Essayons de**:
- créer un nouveau `user`
- fournir des mauvaises données au formulaire pour voir comment elles sont gérées (mauvais format d'email, qui correspond déjà à un `user`...)



---

# Gestion d'utilisateurs

On peut maintenant créer des nouveaux `users`. Super ! Mais la gestion d'utilisateurs, ce n'est pas que pouvoir créer des comptes. C'est aussi:
- pouvoir se connecter
- rester connecter d'une page à l'autre
- restreindre l'accès à certaines pages seulement si on est connecté.e (dans notre cas: modifier la BDD seulement quand on est connecté.e).

TODO: trucs à expliquer:
- utilisation de flask_login:
    - dans `app`, `login_manager`
    - dans `models/users/`, ajout de `UserMixin` à `User`
    - présenter rapidement les trucs utiles: `current_user`, `login_user()`, `logout_user()`, `@login_required`
- `User.login()`
- template `user_login.html`

## La gestion d'utilisateurs: Flask-Login

Flask-Login est un plugin qui gère la connexion d'utilisateurs. Pour l'utiliser, on doit:
- configurer un `LoginManager`
- augmenter `User` pour le rendre compatible avec Flask-Login

### Configurer le `LoginManager`

La première étape, c'est de compléter [`app/app.py`](./apps/s5/login/app/app.py):

```py
from flask_login import LoginManager

app = Flask(
    APP_NAME,
    template_folder=DIR_TEMPLATES, 
    static_folder=DIR_STATICS
)
# j'omets le reste de la config...
login_manager = LoginManager()
login_manager.init_app(app)

from app.routes import generic, users
```

`login_manager` est notre gestionnaire de connexions et il est totalement intégré à notre appli Flask.

### Configurer l'`User`

#### Augmenter `User`: `UserMixin`

Flask-Login a besoin que le modèle pour nos `Users` ait quelques propriétés bien définies pour fonctionner: `is_authenticated`, `is_active`, `is_anonymous`, `get_id` (qui permet de récupérer l'ID de l'utilisateur).

**Pas besoin de les définir à la main**: Flask-Login offre un `UserMixin` qui rajoute ces méthodes à notre modèle `User`. Pour ajouter le mixin, on modifie [`app/models/users.py`](./apps/s5/login/app/models/users.py):

```py
from flask_login import UserMixin

class User(db.Model, UserMixin):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
```

Et c'est tout ! Pour rappel, `User(db.Model, UserMixin)` signifie que **`User` hérite à la fois de `db.Model` et de `UserMixin`**. `UserMixin` définie les propriétés `is_authenticated`, `is_active`, `is_anonymous`, `get_id`m donc celles-ci deviennent accessibles depuis notre `User` (par exemple: `user.get_id()`).

#### Définir un `user_loader`

Notre `LoginManager` défini dans `app/app.py` a besoin de pouvoir accéder à l'utilisateur actuellement connecté. **On définit donc une fonction qui permet d'accéder à un `User` par son ID**, toujours dans [`app/models/users.py`](./apps/s5/login/app/models/users.py):

```py
from app.app import login_manager

# ce décorateur signifie que la fonction `load_user` sera utilisée par le LoginManager pour accéder à l'user connecté.e
@login_manager.user_loader
def load_user(id_user: str):
    id_user = int(id_user)
    return db.session.get(User, id_user)
```

### Ce qui devient possible

Et voilà, Flask-Login est configuré.

Flask-Login offre des choses bien utiles pour nous:
- une variable `current_user`, qui stocke l'`user` actuellement connecté.e dans une session (si il y a un `user` connecté)
- deux fonctions `login_user()` et `logout_user()` qui permettent de connecter/déconnecter un.e `user`.
- un décorateur `@login_required` qui permet de limiter l'accès à une route Flask aux utilisateur.ice.s connecté.es. 

## Login: se connecter

La logique à implémenter: 
- **une route** permet d'accéder à un formulaire pour se connecter
- pour se connecter à un compte existant, **l'utilisateur.ice fournit un mail et un mot de passe**
- **notre application vérifie** si ils correspondent à un `User` dans la base de données
- si oui, **on connecte l'utilisateur**

### Formulaire Flask

Ensemble, ajoutons **un formulaire Flask** pour se connecter. Pour rappel, on se connecte en utilisant son mail et son mdp (on peut utiliser `UserCreateForm` défini plus haut pour référence).


In [14]:
# le code ici

(La solution est dans `apps/s5/login/app/models/forms.py`)

### Templates

Dans notre template HTML pour créer des `Users`, notre formulaire contenait un `<form>` dans lequel on bouclait sur tous les champs du formulaire `UserCreateForm` pour générer du HTML. Ici, on veut suivre la même logique. Plutôt que de dupliquer le HTML de `user_create.html` dans une nouvelle page, **on extraire cette logique et la mettre dans le composant `form_fields.html`**. 

```txt
templates/
├── base.html
├── includes
│   └── form_fields.html   # composant contenant le contenu de formulaires HTML
└── pages
    ├── user_create.html   # page contenant un formulaire HTML pour créer un utilisateur. Inclut `form_fields.html`
    └── user_login.html    # page contenant un formulaire HTML pour se connecter. Inclut `form_fields.html`
```

#### Un composant pour le contenu des formulaires: `form_fields.html`:

Voici le contenu de [`app/templates/includes/form_fields.html`](./apps/s5/login/app/templates/includes/form_fields.html):

```html
{{ form.csrf_token }} 
<dl>
    {% for field in form %}
        {% if field.type != 'CSRFTokenField' %}
            <dt>
                {{ field.label }}
                {% if field.flags.required %}*{% endif %}
            </dt>
            <dd>
                {{ field }}
                {% if field.errors %}
                    <ul class="errors">
                        {% for error in field.errors %}
                            <li>{{ error }}</li>
                        {% endfor %}
                    </ul>
                {% endif %}
            </dd>
        {% endif %}
    {% endfor %}
</dl>
<input type="submit" value="{{ submit_label }}" class="button is-rounded negative">
```

**Remarques**:
- `form_fields.html` contient **exactement ce qui était avant dans `user_create.html`**
- le template a besoin de **2 variables**: 
    - `form`, un formulaire WTForms
    - `submit_label`, le texte à afficher dans l'`<input>` submit.

#### Utiliser `form_fields.html` pour créer des formulaires

Voici comment on l'utilise dans [`./app/templates/pages/user_login.html`](./apps/s5/login/app/templates/pages/user_login.html).

```html
{% extends "base.html" %}

{% block title_extra %}| Se connecter{% endblock %}

{% block main_content %}
    <h1 class="title">Se connecter</h1>

    <!-- comment interprétez vous le contenu de `form` ? -->
    <form method="POST" action="{{ url_for('user_login') }}">
        {% with form=form, submit_label="Connexion" %} 
            {% include "includes/form_fields.html" %}
        {% endwith %}
    </form>
{% endblock %}
```

Et allez aussi voir comment on l'utilise dans [`./app/templates/pages/user_create.html`](./apps/s5/login/app/templates/pages/user_create.html). Que remarquez vous ?

On a maintenant configuré Flask-Login et mis en place les templates pour se connecter ! Il faut maintenant **ajouter la logique côté base de données et une route pour se connecter**.

### Identifier un `User`

**On augmente `User` en ajoutant la méthode `get_user_by_credentials`**. Elle vérifier si un `user` existe pour le mail et le MDP fournis. Voilà le processus:
- l'utilisateur.ice fournit son mail et son mot de passe
- on hashe le mot de passe
- on vérifie qu'il y a une ligne dans la base de données avec le bon mail et le bon hash de mot de passe.
- si oui, on retourne le `user`, sinon on retourne `None`.

```py
class User(db.Model, UserMixin):
    __tablename__ = "user"

    id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    user_name: Mapped[str]
    user_mail: Mapped[str]
    user_password: Mapped[str]
    
    @staticmethod
    def create_user(user_mail: str, user_name: str, user_password: str) -> Tuple[bool, Union["User", List[str]]]:
        ...
    
    @staticmethod
    def get_user_by_credentials(user_mail: str, user_password: str) -> Optional["User"]:
        """
        identifier un User par son mail et son mdp. 

        :returns: l'User si le mail et mdp sont valides, None sinon 
        """
        # retourne soit un `User`, soit None 
        user = db.session.execute(
            db.select(User).filter(User.user_mail == user_mail)
        ).scalars().first()
        # on vérifie que le `User` avec ce mail a bien le bon mot de passe
        if user and check_password_hash(user.user_password, user_password):
            return user
        # sinon, on retourne None
        return None
```

### Une route pour se connecter

Pour finir, on ajoute une route `/user/connexion` pour se connecter.

Voici la route dans [`app/routes/users.py`](./apps/s5/login/app/routes/users.py):

```py
from flask_login import login_user

# comment interprétez vous ce que fait cette route ?
@app.route("/user/connexion/", methods=["GET", "POST"])
def user_login():
    form = UserLoginForm()

    if current_user.is_authenticated:
        flash("Vous êtes déjà connecté.e", "success")
        return redirect("/")

    if form.validate_on_submit():
        user_mail = form.user_mail.data
        user_password = form.user_password.data
        user = User.get_user_by_credentials(
            user_mail=user_mail,
            user_password=user_password
        )
        if user:
            flash("Vous êtes maintenant connecté.e", "success")
            login_user(user)
            return redirect("/")
        else:
            flash("Identifiants incorrects", "error")
            return render_template("pages/user_login.html", form=form, app_name=APP_NAME)
        
    return render_template("pages/user_login.html", form=form, app_name=APP_NAME)
```

## Logout: se déconnecter

De la même manière que l'on a créé une route pour se connecter, on en crée une pour se déconnecter. C'est bien plus simple: on veut une simple requête `GET` à `user/deconnexion/`, et cette requête exécute la fonction Flask-Login `logout_user`:

```py
@app.route("/user/deconnexion/")
@login_required
def user_logout():
    logout_user()
    flash("Vous êtes déconnecté.e")
    return redirect("/")
```

Allons maintenant voir le résultat.

> **On lance l'application**:
> ```py
> python apps/s5/login/main.py
> ```